# FFT Features Experiment

**Goal:** Test whether adding frequency-domain (FFT) features to Config A improves
Model 2's 11-class fault classification, measured by session-aware CV F1-macro.

**Baseline to beat:** Phase 1 Config A = **0.333 ± 0.076**

**Design:** Standard Model 2 (11 fault classes, Normal excluded from scoring),
same CatBoost + StratifiedGroupKFold protocol as Phase 1, for a fair comparison.
The only change is the feature set: Config A (64) vs Config A + FFT (144).

In [1]:
# =============================================================
# STEP 1 - Setup & Load
# =============================================================
import os
import json
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

BASE = r"C:\Users\DELL\Desktop\kpi_rag\Models Improvement"
DATA_DIR = BASE + r"\Data Splits"

# --- Feature schema: confirm Config A range (0..64) ---
with open(os.path.join(DATA_DIR, "feature_schema.json")) as f:
    feature_schema = json.load(f)

_cfg = feature_schema["configs"]["A"]           # [0, 64]
CONFIG_A = np.arange(_cfg[0], _cfg[1])
NORMAL_LABEL = feature_schema.get("normal_label", "Normal")
print(f"Config A features: {len(CONFIG_A)}  (expected 64)")
assert len(CONFIG_A) == 64

# --- Feature matrix (train partition, corrected 584-dim) ---
train_X = np.load(os.path.join(DATA_DIR, "train_X_sessionsplit_fixed.npy"), allow_pickle=True)
train_y_str = np.load(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), allow_pickle=True)

# --- Split indices + session ids ---
si_train = np.load(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), allow_pickle=True).astype(int)
session_id_by_row = np.load(os.path.join(DATA_DIR, "session_id_by_row.npy"), allow_pickle=True)
train_sessions_all = session_id_by_row[si_train]

# --- FFT features (full 32,000 rows, original CSV order) ---
fft_full = np.load(os.path.join(DATA_DIR, "fft_features.npy"), allow_pickle=True)

print("train_X:", train_X.shape)
print("train_y:", train_y_str.shape)
print("fft_full:", fft_full.shape)
print("si_train:", si_train.shape)
print("train_sessions_all:", train_sessions_all.shape)


Config A features: 64  (expected 64)
train_X: (25688, 584)
train_y: (25688,)
fft_full: (32000, 80)
si_train: (25688,)
train_sessions_all: (25688,)


In [2]:
# =============================================================
# STEP 2 - Align FFT features to the TRAIN partition
# =============================================================
# fft_full has 32,000 rows in original CSV order.
# train_X is the train partition. si_train maps train rows -> original rows.
# So fft_full[si_train] gives FFT rows in the SAME order as train_X.

fft_train = fft_full[si_train]
print("fft_train:", fft_train.shape, " (should match train_X rows)")
assert fft_train.shape[0] == train_X.shape[0], "Row mismatch - FFT and train_X are not aligned!"
print("Alignment OK: FFT rows match train_X rows.")


fft_train: (25688, 80)  (should match train_X rows)
Alignment OK: FFT rows match train_X rows.


In [3]:
# =============================================================
# STEP 3 - Build the two feature sets (anomalous windows only)
# =============================================================
# Model 2 trains on the 11 fault classes only (Normal excluded),
# exactly like Phase 1.

anom_mask = train_y_str != NORMAL_LABEL

# Config A only (baseline reproduction)
X_configA = train_X[anom_mask][:, CONFIG_A]

# Config A + FFT (the experiment)
X_fft = fft_train[anom_mask]
X_combined = np.concatenate([X_configA, X_fft], axis=1)

y_anom = train_y_str[anom_mask]
sessions_anom = train_sessions_all[anom_mask]

print("X_configA :", X_configA.shape, " (64 features)")
print("X_combined:", X_combined.shape, " (64 + 80 = 144 features)")
print("y_anom    :", y_anom.shape)
print("classes   :", len(np.unique(y_anom)))
print("\nClass support:")
print(pd.Series(y_anom).value_counts().sort_values())


X_configA : (935, 64)  (64 features)
X_combined: (935, 144)  (64 + 80 = 144 features)
y_anom    : (935,)
classes   : 11

Class support:
High Network Congestion (Sudden Spike)        37
Resource Allocation Bugs                      43
Faulty Handover Algorithm (Too Frequent)      45
High Network Congestion (Gradual Buildup)     56
Faulty RF Filters (Temporal)                  67
Doppler Shift (Severe)                        77
Co-Channel Interference (Severe)              84
Antenna Failure                               98
Buffer Overflow (Gradual Buildup)            117
Co-Channel Interference (Mild)               126
Jamming                                      185
Name: count, dtype: int64


In [5]:
# =============================================================
# STEP 4 - Session-aware CV (identical protocol to Phase 1)
# =============================================================
def make_clf(class_weight=None):
    return CatBoostClassifier(
        random_seed=SEED, verbose=0, thread_count=4,
        class_weights=class_weight,
    )

def run_cv(X, y, sessions, label, k=5):
    n_groups = len(np.unique(sessions))
    actual_k = min(k, n_groups)
    cv = StratifiedGroupKFold(n_splits=actual_k, shuffle=True, random_state=SEED)

    fold_f1s = []
    for fold_i, (tr_i, val_i) in enumerate(cv.split(X, y, groups=sessions)):
        X_tr, X_val = X[tr_i], X[val_i]
        y_tr, y_val = y[tr_i], y[val_i]

        counts = Counter(y_tr)
        weights = {c: len(y_tr) / (len(counts) * n) for c, n in counts.items()}

        clf = make_clf(class_weight=weights)
        clf.fit(X_tr, y_tr)

        preds = clf.predict(X_val).ravel()
        f1 = f1_score(y_val, preds, average="macro", zero_division=0)
        fold_f1s.append(f1)
        print(f"  Fold {fold_i+1}: F1-macro = {f1:.4f}")

    mean_f1 = np.mean(fold_f1s)
    std_f1 = np.std(fold_f1s)
    ci_low = mean_f1 - 1.96 * std_f1
    ci_high = mean_f1 + 1.96 * std_f1
    print(f"\n{label}")
    print(f"  CV F1-macro: {mean_f1:.4f} +/- {std_f1:.4f}")
    print(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
    return mean_f1, std_f1, (ci_low, ci_high)


In [6]:
# =============================================================
# STEP 5 - Baseline: Config A only (reproduce Phase 1)
# =============================================================
print("=== BASELINE: Config A (64 features) ===\n")
base_mean, base_std, base_ci = run_cv(X_configA, y_anom, sessions_anom, "Config A baseline")
print("\nPhase 1 reported: 0.333 +/- 0.076")
print("(This should land close to 0.333 - it confirms the setup is correct.)")


=== BASELINE: Config A (64 features) ===

  Fold 1: F1-macro = 0.2132
  Fold 2: F1-macro = 0.2749
  Fold 3: F1-macro = 0.2909
  Fold 4: F1-macro = 0.2429
  Fold 5: F1-macro = 0.4491

Config A baseline
  CV F1-macro: 0.2942 +/- 0.0819
  95% CI: [0.1336, 0.4548]

Phase 1 reported: 0.333 +/- 0.076
(This should land close to 0.333 - it confirms the setup is correct.)


In [7]:
# =============================================================
# STEP 6 - Experiment: Config A + FFT (144 features)
# =============================================================
print("=== EXPERIMENT: Config A + FFT (144 features) ===\n")
fft_mean, fft_std, fft_ci = run_cv(X_combined, y_anom, sessions_anom, "Config A + FFT")


=== EXPERIMENT: Config A + FFT (144 features) ===

  Fold 1: F1-macro = 0.3579
  Fold 2: F1-macro = 0.2928
  Fold 3: F1-macro = 0.3098
  Fold 4: F1-macro = 0.2168
  Fold 5: F1-macro = 0.5435

Config A + FFT
  CV F1-macro: 0.3441 +/- 0.1095
  95% CI: [0.1295, 0.5588]


In [9]:
# =============================================================
# STEP 7 - Compare & Verdict
# =============================================================
delta = fft_mean - base_mean
print("=" * 55)
print(f"{'Configuration':<28}{'CV F1-macro':<18}")
print("-" * 55)
print(f"{'Config A (baseline)':<28}{base_mean:.4f} +/- {base_std:.4f}")
print(f"{'Config A + FFT':<28}{fft_mean:.4f} +/- {fft_std:.4f}")
print("-" * 55)
print(f"{'Delta (FFT - baseline)':<28}{delta:+.4f}")
print("=" * 55)

# CI overlap check
overlap = not (fft_ci[0] > base_ci[1] or fft_ci[1] < base_ci[0])
print(f"\nConfidence intervals overlap: {overlap}")
if delta > 0 and not overlap:
    print("VERDICT: FFT improves performance (statistically distinct). PROMOTE.")
elif delta > 0 and overlap:
    print("VERDICT: FFT shows a positive trend but CIs overlap - not conclusive.")
else:
    print("VERDICT: FFT does not improve performance. Do not promote.")

# Save results
results = {
    "config_A_cv_f1_mean": float(base_mean),
    "config_A_cv_f1_std": float(base_std),
    "config_A_fft_cv_f1_mean": float(fft_mean),
    "config_A_fft_cv_f1_std": float(fft_std),
    "delta": float(delta),
    "ci_overlap": bool(overlap),
    "phase1_baseline": 0.333,
}
with open(os.path.join(DATA_DIR, "fft_experiment_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: fft_experiment_results.json")


Configuration               CV F1-macro       
-------------------------------------------------------
Config A (baseline)         0.2942 +/- 0.0819
Config A + FFT              0.3441 +/- 0.1095
-------------------------------------------------------
Delta (FFT - baseline)      +0.0499

Confidence intervals overlap: True
VERDICT: FFT shows a positive trend but CIs overlap - not conclusive.

Saved: fft_experiment_results.json
